# OpenClaw Overall Harness Lab

This Colab notebook uses OpenClaw itself to show the difference between a raw model probe and a full agent turn.

You will run two lanes:

1. **DeepSeek API lane**: stable live demo path.
2. **Ollama lane**: free local-model path inside Colab.

The core idea: a model is not an agent. The runtime or harness executes a prepared agent turn.

## 0. Runtime notes

This notebook is designed for Google Colab. It does not modify your local machine.

The DeepSeek lane requires a `DEEPSEEK_API_KEY`. The Ollama lane downloads and serves a small model inside the Colab runtime. Small local models may pass the model probe but fail a full tool-using agent turn; that is an expected observation, not a lab failure.

In [ ]:
import os
import pathlib
import shutil
import subprocess

REPO_URL = "https://github.com/SleepyLGod/openclaw-teaching.git"
REPO_DIR = pathlib.Path("/content/openclaw-teaching")
LAB_DIR = REPO_DIR / "labs" / "overall-harness"
WORKSPACE_SRC = LAB_DIR / "workspace"
OPENCLAW_WORKSPACE = pathlib.Path.home() / ".openclaw" / "workspace"

def section(title: str) -> None:
    line = "=" * len(title)
    print(f"\n{line}\n{title}\n{line}")

def show_file(path: pathlib.Path) -> None:
    print(f"\n--- {path} ---")
    print(path.read_text(encoding="utf-8"))

def run(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    print(f"\n$ {cmd}")
    proc = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
    return proc


## 1. Clone the teaching repo

The GitBook page explains the ideas. The repo contains the runnable lab files.

In [ ]:
section("1. Clone the teaching repo")
print("Observation target: load the lab materials that define the OpenClaw workspace.")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")

print("Lab directory:", LAB_DIR)
print("Workspace source directory:", WORKSPACE_SRC)
print("Workspace files and contents before copying into OpenClaw:")
for path in sorted(WORKSPACE_SRC.iterdir()):
    if path.is_file():
        show_file(path)


## 2. Install OpenClaw

The installer handles Node and OpenClaw setup. We skip interactive onboarding because the notebook configures providers explicitly later.

In [ ]:
section("2. Install OpenClaw")
print("Observation target: the CLI exists before we test any model or full agent turn.")

run("curl -fsSL https://openclaw.ai/install.sh | bash -s -- --no-onboard", check=False)

# Make common install locations visible to this notebook process.
os.environ["PATH"] = ":".join([
    str(pathlib.Path.home() / ".local" / "bin"),
    str(pathlib.Path.home() / ".openclaw" / "bin"),
    os.environ.get("PATH", ""),
])

run("command -v openclaw || true", check=False)
run("openclaw --version", check=False)
run("openclaw doctor", check=False)


## 3. Prepare the OpenClaw workspace

These files are intentionally small. They make the effect of workspace/bootstrap context visible.

In [ ]:
section("3. Prepare the OpenClaw workspace")
print("Observation target: workspace/bootstrap files become runtime inputs for the full agent turn.")

OPENCLAW_WORKSPACE.mkdir(parents=True, exist_ok=True)
for src in WORKSPACE_SRC.iterdir():
    if src.is_file():
        shutil.copy2(src, OPENCLAW_WORKSPACE / src.name)

print("OpenClaw workspace target:", OPENCLAW_WORKSPACE)
print("Copied bootstrap/workspace files:")
for path in sorted(OPENCLAW_WORKSPACE.glob("*")):
    if path.is_file() and path.name in {"SOUL.md", "USER.md", "AGENTS.md", "notes.txt"}:
        show_file(path)


## 4. DeepSeek API lane

This is the stable demo path. Put your API key into Colab secrets or paste it when prompted.

The provider is `deepseek`; the default model for this lab is `deepseek/deepseek-v4-flash`.

In [ ]:
section("4. DeepSeek API provider setup")
print("Observation target: configure the stable API provider without printing the API key.")

import getpass

if not os.environ.get("DEEPSEEK_API_KEY"):
    try:
        from google.colab import userdata
        secret = userdata.get("DEEPSEEK_API_KEY")
    except Exception:
        secret = None

    if secret:
        os.environ["DEEPSEEK_API_KEY"] = secret
        print("Loaded DEEPSEEK_API_KEY from Colab Secrets.")
    else:
        os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("Enter DEEPSEEK_API_KEY: ")
else:
    print("DEEPSEEK_API_KEY already exists in the environment.")

run('openclaw onboard --non-interactive --mode local --auth-choice deepseek-api-key --deepseek-api-key "$DEEPSEEK_API_KEY" --skip-health --accept-risk', check=False)
run("openclaw models list --all --provider deepseek", check=False)


### 4A. DeepSeek model probe

This checks the provider/model path only.

In [ ]:
section("4A. DeepSeek model probe")
DEEPSEEK_MODEL = "deepseek/deepseek-v4-flash"
print("Provider: deepseek")
print("Model reference:", DEEPSEEK_MODEL)
print("Expected observation: this checks the provider/model path only, not the full harness turn.")
run(f"openclaw infer model run --local --model {DEEPSEEK_MODEL} --prompt 'Reply exactly: COURSE_OK' --json", check=False)


### 4B. DeepSeek full agent turn

This runs through the OpenClaw agent runtime. The runtime sees the workspace, bootstrap instructions, session key, context, and tool surface.

In [ ]:
section("4B. DeepSeek full agent turn")
print("This is now a full runtime/harness turn.")
print("Session key: harness-deepseek-a")
print("Workspace file expected to matter: notes.txt")
run("openclaw agent --local --session-key harness-deepseek-a --message 'Read notes.txt and answer in one sentence: what does this lab teach?'", check=False)

print("")
print("This second full turn uses a fresh session key to observe bootstrap instructions without stale history.")
print("Session key: harness-deepseek-b")
print("Workspace files expected to matter: USER.md and SOUL.md")
run("openclaw agent --local --session-key harness-deepseek-b --message 'How should you address me, and what style should you use?'", check=False)


## 5. Ollama lane

This is the free local-model path inside Colab. It may be slower and less reliable than the API lane.

The goal is still the same: compare a narrow model probe with a full agent turn.

In [ ]:
section("5. Install and start Ollama")
print("Observation target: start a local model server inside the Colab runtime.")

run("curl -fsSL https://ollama.com/install.sh | sh", check=False)

# Start Ollama in the background if it is not already running.
run("pgrep -x ollama >/dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)", check=False)
run("sleep 5 && curl -s http://127.0.0.1:11434/api/tags || true", check=False)


### 5A. Pull a small model

`llama3.2:3b` is small enough for a Colab demo. You can replace it with a stronger model if your runtime has enough resources.

In [ ]:
section("5A. Pull a small Ollama model")
OLLAMA_MODEL_NAME = "llama3.2:3b"
OLLAMA_MODEL_REF = f"ollama/{OLLAMA_MODEL_NAME}"
print("Provider: ollama")
print("Model reference:", OLLAMA_MODEL_REF)
print("Expected observation: model download succeeds and OpenClaw can inspect the Ollama provider.")
run(f"ollama pull {OLLAMA_MODEL_NAME}", check=False)
run("openclaw models list --all --provider ollama", check=False)


### 5B. Ollama model probe

This should succeed before you try a full agent turn.

In [ ]:
section("5B. Ollama model probe")
print("Provider: ollama")
print("Model reference:", OLLAMA_MODEL_REF)
print("Expected observation: this checks local model inference before trying a full agent turn.")
run(f"openclaw infer model run --local --model {OLLAMA_MODEL_REF} --prompt 'Reply exactly: OLLAMA_OK' --json", check=False)


### 5C. Ollama full agent turn

Small local models can fail when the full agent context and tool surface are present. If the model probe passed but this full agent turn fails, the useful observation is: a raw model path can work while the full runtime/harness turn is too demanding for that model.


In [ ]:
section("5C. Ollama full agent turn")
print("This is now a full runtime/harness turn with a small local model.")
print("Session key: harness-ollama-a")
print("Workspace file expected to matter: notes.txt")
print("If this fails after the probe passed, record it as model/tool-use limitation under full runtime load.")
run(f"openclaw agent --local --model {OLLAMA_MODEL_REF} --session-key harness-ollama-a --message 'Read notes.txt and answer in one sentence: what does this lab teach?'", check=False)


## 6. Observe the difference

Answer these questions in your notes:

1. Which command was only a provider/model probe?
2. Which command ran a full agent turn?
3. What provider and model did you use in each lane?
4. Which workspace files shaped the agent response?
5. Why can the model probe pass while the full agent turn fails?
6. Which parts are harness/runtime concerns?
7. Which parts should be deferred to the Guardrails chapter?

## 7. Key takeaway

A model returns text. A harness runs an agent turn.

The agent loop is the runtime's flow. Constraints are policy surfaces the runtime must respect. The harness is the implementation that owns the prepared turn.